In [2]:
import re

def extract_numbers(text):
    """Pull every number out of a text answer."""
    # Season notations like 24/25 or 2024-25 are not stat claims — drop them first
    text = re.sub(r"\b\d{2,4}[/-]\d{2,4}\b", " ", text)
    raw = re.findall(r"\d[\d,]*\.?\d*", text)
    return [float(n.replace(",", "")) for n in raw]

In [3]:
# quick test
extract_numbers("Saka scored six goals and +10 assists in 1,729 minutes (0.31 per 90) in 24/25 season.")

[10.0, 1729.0, 0.31, 90.0]

In [4]:
import math

def verify_numbers(answer_text, source_record):
    """Check every number in the answer against the source data.

    source_record: the dict returned by get_player_stats
    Returns a verdict dict — never raises.
    """
    claimed = extract_numbers(answer_text)

    # Flatten all numeric values from all team records into one pool
    truth = []
    for rec in source_record.get("records", []):
        for v in rec["stats"].values():
            if isinstance(v, (int, float)):
                truth.append(float(v))

    unverified = []
    for num in claimed:
        ok = any(math.isclose(num, t, rel_tol=0.01) for t in truth)
        if not ok:
            unverified.append(num)

    return {
        "verdict": "pass" if not unverified else "fail",
        "claimed": claimed,
        "unverified": unverified,
    }

In [7]:
import sys; sys.path.append("..")
from data.loader import get_player_stats

source = get_player_stats("Saka")

# ① a correct answer (from today's real run)
good = "Bukayo Saka has scored 6 goals for Arsenal this season (24/25)."
print(verify_numbers(good, source))

# ② a hallucinated answer (fabricated numbers)
bad = "Saka scored 93 goals and 50 assists this season."
print(verify_numbers(bad, source))

[07/10/26 22:47:44] INFO     No custom team name replacements found. You can configure these in       ]8;id=16381897;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=16381898;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/jade/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=16381904;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=16381905;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/jade/soccerdata/config/league_dict.json.                                       

                    INFO     Saving cached data to /Users/jade/soccerdata/data/FBref                 ]8;id=16381912;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=16381913;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\

{'verdict': 'pass', 'claimed': [6.0], 'unverified': []}
{'verdict': 'fail', 'claimed': [93.0, 50.0], 'unverified': [93.0, 50.0]}


<br><br> << Verify numbers >>

In [5]:
import math

def verify_numbers(answer_text, source_record):
    """Check every number in the answer against the source data.

    source_record: the dict returned by get_player_stats
    Returns a verdict dict — never raises.
    """
    claimed = extract_numbers(answer_text)

    # Flatten all numeric values from all team records into one pool
    truth = []
    for rec in source_record.get("records", []):
        for v in rec["stats"].values():
            if isinstance(v, (int, float)):
                truth.append(float(v))

    unverified = []
    for num in claimed:
        ok = any(math.isclose(num, t, rel_tol=0.01) for t in truth)
        if not ok:
            unverified.append(num)

    return {
        "verdict": "pass" if not unverified else "fail",
        "claimed": claimed,
        "unverified": unverified,
    }

In [11]:
import sys; sys.path.append("..")
from data.loader import get_player_stats

source = get_player_stats("Saka")

# ① a correct answer (from today's real run)
good = "Bukayo Saka has scored 6 goals for Arsenal this season (24/25)."
print(verify_numbers(good, source))

# ② a hallucinated answer (fabricated numbers)
bad = "Saka scored 93 goals and 50 assists this season."
print(verify_numbers(bad, source))

{'verdict': 'pass', 'claimed': [6.0], 'unverified': []}
{'verdict': 'fail', 'claimed': [93.0, 50.0], 'unverified': [93.0, 50.0]}


In [14]:
source_r = get_player_stats("Rashford")
ans_r = "Rashford scored 4 goals for Manchester Utd and 2 goals for Aston Villa, 6 in total."
print(verify_numbers(ans_r, source_r))

{'verdict': 'fail', 'claimed': [4.0, 2.0, 6.0], 'unverified': [6.0]}


In [15]:
hallucinated = ("Rashford has 4 goals in 978 minutes for Manchester United. "
                "The Aston Villa record appears to be a database inconsistency.")
print(verify_numbers(hallucinated, source_r))

{'verdict': 'pass', 'claimed': [4.0, 978.0], 'unverified': []}


In [8]:
from judges.deterministic import verify_numbers as judge_from_module
print(judge_from_module(bad, source))   # fail 나오면 이사 성공

{'verdict': 'fail', 'claimed': [93.0, 50.0], 'unverified': [93.0, 50.0]}
